1. Import Library

In [34]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

2. Load Dataset

In [35]:
# ============================================================
# LOAD DATASET
# ============================================================

DATA_PATH = "../data/processed/cases.csv"

df = pd.read_csv(DATA_PATH)

print("Jumlah kasus :", len(df))

df.head()

Jumlah kasus : 50


,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full
0,1,case_001.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
1,2,case_002.txt,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30",gnomor 1092 k pid 2024 memeriksa perkara tinda...,dinyatakan ditolak dengan perbaikan e menimnba...,1230,8511,gnomor 1092 k pid 2024 memeriksa perkara tinda...
2,3,case_003.txt,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa...",gnomor 1153 pk pid.sus 2024 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,882,6185,gnomor 1153 pk pid.sus 2024 memeriksa perkara ...
3,4,case_004.txt,1171 pk pid.sus 2026,2026-01-05,"pasal 2, pasal 318, pasal 322, pasal 455, pasa...",gnomor 1171 pk pid.sus 2026 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,943,6493,gnomor 1171 pk pid.sus 2026 memeriksa perkara ...
4,5,case_005.txt,11809 k pid.sus 2025,2025-12-11,"pasal 2, pasal 253, pasal 296",gnomor 11809 k pid.sus 2025 memeriksa perkara ...,menolak permohonan kasasi dari pemohon kasasi ...,1354,9340,gnomor 11809 k pid.sus 2025 memeriksa perkara ...


3. Cek Missing Value

In [36]:
# ============================================================
# CEK MISSING VALUE
# ============================================================

df.isnull().sum()

case_id            0
file_name          0
no_perkara         0
tanggal            0
pasal              0
ringkasan_fakta    0
amar_putusan       0
jumlah_kata        0
jumlah_karakter    0
text_full          0
dtype: int64

4. Handle Missing Value

In [37]:
# ============================================================
# HANDLE MISSING VALUE
# ============================================================

df = df.fillna("")

print("Dataset bersih")

Dataset bersih


5. Perbaikan Amar Putusan

In [38]:
# ============================================================
# FIX OUTLIER AMAR PUTUSAN
# ============================================================

df.loc[
    df["case_id"] == 7,
    "amar_putusan"
] = (
    "permohonan peninjauan kembali dinyatakan ditolak "
    "dan putusan yang dimohonkan peninjauan kembali "
    "dinyatakan tetap berlaku"
)

df.loc[
    df["case_id"] == 19,
    "amar_putusan"
] = (
    "permohonan kasasi dinyatakan ditolak "
    "dengan perbaikan"
)

print(
    "Tidak ditemukan :",
    (df["amar_putusan"] == "Tidak Ditemukan").sum()
)

Tidak ditemukan : 0


6. Membuat Amar Singkat

In [39]:
# ============================================================
# AMAR PUTUSAN SINGKAT
# ============================================================

def simplify_amar(text):

    text = str(text).lower()

    if "ditolak dengan perbaikan" in text:
        return "ditolak dengan perbaikan"

    elif "dinyatakan ditolak" in text:
        return "dinyatakan ditolak"

    elif "menolak permohonan" in text:
        return "menolak permohonan"

    elif "mengabulkan permohonan" in text:
        return "mengabulkan permohonan"

    elif "mengadili" in text:
        return "mengadili"

    return "lainnya"


df["amar_singkat"] = df["amar_putusan"].apply(
    simplify_amar
)

df[
    [
        "no_perkara",
        "amar_singkat"
    ]
].head()

,no_perkara,amar_singkat
0,1013 k pid.sus 2026,mengadili
1,1092 k pid 2024,ditolak dengan perbaikan
2,1153 pk pid.sus 2024,dinyatakan ditolak
3,1171 pk pid.sus 2026,dinyatakan ditolak
4,11809 k pid.sus 2025,menolak permohonan


CELL 7 — Membuat Retrieval Text

Sesuai konsep CBR.

Gunakan:

Ringkasan Fakta + Pasal

karena itu identitas utama sebuah kasus.

In [40]:
# ============================================================
# RETRIEVAL TEXT
# ============================================================

df["retrieval_text"] = (

    df["pasal"].astype(str)

    + " "

    + df["amar_singkat"].astype(str)

    + " "

    + df["text_full"].astype(str).str[:5000]

)

8. Train Test Split

In [41]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42

)

print("Train :", len(train_df))
print("Test  :", len(test_df))

Train : 40
Test  : 10


9. TF-IDF Vectorization

In [42]:
# ============================================================
# TF-IDF VECTORIZATION
# ============================================================

vectorizer = TfidfVectorizer(

    max_features=5000,

    stop_words=None

)

X_train = vectorizer.fit_transform(

    train_df["retrieval_text"]

)

X_test = vectorizer.transform(

    test_df["retrieval_text"]

)

print("Shape Train :", X_train.shape)
print("Shape Test :", X_test.shape)

Shape Train : (40, 3057)
Shape Test : (10, 3057)


10. Similarity Matrix

In [43]:
# ============================================================
# SIMILARITY MATRIX
# ============================================================

similarity_matrix = cosine_similarity(

    X_train,

    X_train

)

print(similarity_matrix.shape)

(40, 40)


11. Fungsi Retrieval

In [44]:
# ============================================================
# RETRIEVE FUNCTION
# ============================================================

def retrieve(

    query,

    k=5

):

    query_vector = vectorizer.transform(

        [query]

    )

    similarity_scores = cosine_similarity(

        query_vector,

        X_train

    )[0]

    top_idx = np.argsort(

        similarity_scores

    )[::-1][:k]

    results = train_df.iloc[top_idx][

        [

            "case_id",

            "no_perkara",

            "tanggal",

            "pasal",

            "amar_singkat"

        ]

    ].copy()

    results["similarity"] = (

        similarity_scores[top_idx]

    )

    return results

12. Query Test

In [45]:
# ============================================================
# QUERY TEST
# ============================================================

query_case = test_df.iloc[0]

print("NO PERKARA")
print(query_case["no_perkara"])

print("\nPASAL")
print(query_case["pasal"])

print("\nAMAR PUTUSAN")
print(query_case["amar_singkat"])

NO PERKARA
1684 k pid.sus 2026

PASAL
pasal   55, pasal 10, pasal 17, pasal 345, pasal 4, pasal 432, pasal 53, pasal 55

AMAR PUTUSAN
ditolak dengan perbaikan


13. Hasil Retrieval

In [46]:
# ============================================================
# TOP 5 SIMILAR CASES
# ============================================================

retrieve(

    query=query_case["retrieval_text"],

    k=5

)

,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.807804
12,13,1683 k pid.sus 2026,2026-02-11,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.789665
15,16,1930 k pid.sus 2026,2026-02-06,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81",ditolak dengan perbaikan,0.516509
16,17,1935 k pid.sus 2026,2025-10-28,"pasal 2, pasal 20, pasal 3, pasal 361, pasal 4...",ditolak dengan perbaikan,0.501086
7,8,12069 k pid.sus 2025,2025-09-25,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 81",mengadili,0.496070


14 .Evaluasi 10 Query

In [47]:
# ============================================================
# EVALUASI AWAL
# ============================================================

for i in range(min(10, len(test_df))):

    query = test_df.iloc[i]

    print("="*100)

    print(
        "QUERY:",
        query["no_perkara"]
    )

    hasil = retrieve(

        query["retrieval_text"],

        k=3

    )

    display(hasil)

QUERY: 1684 k pid.sus 2026


,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.807804
12,13,1683 k pid.sus 2026,2026-02-11,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.789665
15,16,1930 k pid.sus 2026,2026-02-06,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81",ditolak dengan perbaikan,0.516509


QUERY: 4132 k pid.sus 2026


,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
34,35,363 k pid 2026,2026-03-05,"pasal 296, pasal 2, pasal 29, pasal 296, pas...",mengadili,0.671703
31,32,2854 k pid.sus 2026,2025-10-16,"pasal 2, pasal 296, pasal 361, pasal 455",ditolak dengan perbaikan,0.594839
15,16,1930 k pid.sus 2026,2026-02-06,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81",ditolak dengan perbaikan,0.580764


QUERY: 278 k pid.sus 2026


...

,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.480333
7,8,12069 k pid.sus 2025,2025-09-25,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 81",mengadili,0.469064
31,32,2854 k pid.sus 2026,2025-10-16,"pasal 2, pasal 296, pasal 361, pasal 455",ditolak dengan perbaikan,0.458660


QUERY: 306 pk pid.sus 2026


,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
10,11,151 k pid.sus 2026,2026-01-21,"pasal 20, pasal 4, pasal 55, pasal 72, pasal 86",menolak permohonan,0.327050
5,6,11815 k pid.sus 2025,2024-11-26,"pasal 55, pasal 12, pasal 17, pasal 2, pasal 55",dinyatakan ditolak,0.312357
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.309890


QUERY: 2000k pid.sus 2026


,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
28,29,255 k pid.sus 2026,2026-01-22,"pasal 11, pasal 197, pasal 2, pasal 20, pasal ...",dinyatakan ditolak,0.695375
23,24,2191 k pid 2025,2025-12-19,"pasal 226, pasal 257, pasal 296",menolak permohonan,0.609504
18,19,1998 k pid.sus 2026,2025-09-24,"pasal 2, pasal 55, pasal 6, pasal 9",ditolak dengan perbaikan,0.441403


15. Top Case untuk Reuse

In [48]:
# ============================================================
# TOP CASE UNTUK REUSE
# ============================================================

top_case = retrieve(
    query_case["retrieval_text"],
    k=1
)

top_case

,case_id,no_perkara,tanggal,pasal,amar_singkat,similarity
11,12,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",ditolak dengan perbaikan,0.807804


16. Membuat File Evaluasi

In [49]:
# ============================================================
# BUILD QUERY JSON
# ============================================================

queries = []

for idx in test_df.index:

    queries.append({

        "query_case_id":

        int(df.loc[idx,"case_id"]),

        "ground_truth":

        int(df.loc[idx,"case_id"])

    })

queries[:5]

[{'query_case_id': 14, 'ground_truth': 14},
 {'query_case_id': 40, 'ground_truth': 40},
 {'query_case_id': 31, 'ground_truth': 31},
 {'query_case_id': 46, 'ground_truth': 46},
 {'query_case_id': 18, 'ground_truth': 18}]

17. Simpan queries.json

In [50]:
# ============================================================
# SAVE QUERY
# ============================================================

with open(

    "../data/eval/queries.json",

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        queries,

        f,

        indent=4

    )

print("queries.json berhasil dibuat")

queries.json berhasil dibuat
